In [1]:
# ============================================================
# BenchRec matcher as generated by ChatGPT: Variation 1 (WORKING)
# - Correctly matches: TRAIN (B_tr↔A_tr), EVAL (B_ev↔A_ev)
# - Prints candidate diagnostics (merge/amount/date/cap)
# - TF-IDF char n-gram cosine + token overlap + superstring rescue
# - Wide amount/date tolerances
# - Calibrated confidence + threshold for TARGET_PRECISION
# - 1-to-1 assignment (each A and B used once)
# - Outputs submission.csv + submission_with_details.csv
# - Evaluates match rate and precision vs solution (subset alloc rule)
# ============================================================

import re
import numpy as np
import pandas as pd
from collections import Counter
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import precision_recall_curve

for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

# ----------------------------
# PATHS (edit these)
# ----------------------------

TRAIN_PATH = "/kaggle/input/benchrec-real-world-cash-reconciliation-dataset/BenchRec_cash_v1.0_train.csv"
EVAL_PATH  = "/kaggle/input/benchrec-real-world-cash-reconciliation-dataset/BenchRec_cash_v1.0_eval.csv"
SOL_PATH = "/kaggle/input/benchrec-real-world-cash-reconciliation-dataset/BenchRec_cash_v1.0_solution.csv"
# ----------------------------
# CONFIG
# ----------------------------
DATE_TOL_DAYS = 10
AMT_ABS_TOL   = 1.00
AMT_PCT_TOL   = 0.0075
AMT_CENT_OFFSETS = [0.00, 0.01, -0.01, 0.02, -0.02]

NGRAM_RANGE = (3, 5)
MIN_DF = 2
MAX_FEATURES = 200_000

MAX_CANDIDATES_PER_B = 200
TARGET_PRECISION = 0.998

# ----------------------------
# HELPERS
# ----------------------------
TOKEN_RE = re.compile(r"[A-Z0-9]+")
STOP = set([
    "USD","EUR","GBP","JPY","AUD","CAD","SGD","HKD","CR","DR","CREDIT","DEBIT",
    "PAYMENT","TRANSFER","BANK","REF","REFERENCE","FROM","TO","THE","AND","FOR",
    "ON","OF","WIRE","ACH","SEPA","SWIFT","FEE"
])

def clean_text(s):
    if s is None:
        return ""
    s = str(s)
    if s.lower() == "nan":
        return ""
    return s.upper()

def norm_id(x):
    if x is None:
        return ""
    s = str(x).strip()
    if s.lower() in ("", "nan"):
        return ""
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def to_float(x):
    try:
        s = str(x).strip()
        if s.lower() in ("", "nan"):
            return np.nan
        return float(s)
    except:
        return np.nan

def to_date(x):
    s = str(x).strip()
    if s.lower() in ("", "nan"):
        return pd.NaT
    return pd.to_datetime(s, errors="coerce").normalize()

def opposite(dc):
    dc = (dc or "").upper().strip()
    if dc == "DR":
        return "CR"
    if dc == "CR":
        return "DR"
    return ""

def extract_tokens(text):
    text = clean_text(text)
    if not text:
        return []
    toks = TOKEN_RE.findall(text)
    out = []
    for t in toks:
        if len(t) < 4:
            continue
        if t in STOP:
            continue
        if any(ch.isdigit() for ch in t) or len(t) >= 8:
            out.append(t)
    seen = set()
    res = []
    for t in out:
        if t not in seen:
            seen.add(t)
            res.append(t)
    return res

def parse_alloc_set(raw):
    if raw is None:
        return set()
    s = str(raw).strip()
    if s.lower() in ("", "nan"):
        return set()
    if s.startswith("[") and s.endswith("]"):
        inner = s[1:-1].strip()
        if not inner:
            return set()
        parts = [p.strip().strip('"').strip("'") for p in inner.split(",") if p.strip()]
        return set(parts)
    return {s.strip('"').strip("'")}

def fmt_alloc_set(alloc_raw):
    s = str(alloc_raw).strip()
    if s.lower() in ("", "nan"):
        return "[]"
    aset = sorted(list(parse_alloc_set(s)))
    if not aset:
        return "[]"
    inner = ",".join([f'"{a}"' for a in aset])
    return f"[{inner}]"

def amount_tol(x):
    if np.isnan(x):
        return np.nan
    return max(AMT_ABS_TOL, AMT_PCT_TOL * abs(x))

def superstring_hits(b_tokens, a_text, cap=20):
    if not b_tokens or not a_text:
        return 0
    c = 0
    for t in b_tokens[:cap]:
        if t and t in a_text:
            c += 1
    return c

def rowwise_cosine(A_mat, B_mat, a_rows, b_rows, chunk=200_000):
    sims = np.zeros(len(a_rows), dtype=float)
    for i in range(0, len(a_rows), chunk):
        sl = slice(i, min(i + chunk, len(a_rows)))
        A_chunk = A_mat[a_rows[sl]]
        B_chunk = B_mat[b_rows[sl]]
        sims[sl] = np.asarray(A_chunk.multiply(B_chunk).sum(axis=1)).ravel()
    return sims

# ----------------------------
# LOAD & SPLIT
# ----------------------------
train = pd.read_csv(TRAIN_PATH, dtype=str)
ev    = pd.read_csv(EVAL_PATH, dtype=str)
sol   = pd.read_csv(SOL_PATH, dtype=str)

A_tr = train[train["A_transactionType"].fillna("") == "A"].copy().reset_index(drop=True)
B_tr = train[train["B_transactionType"].fillna("") == "B"].copy().reset_index(drop=True)
A_ev = ev[ev["A_transactionType"].fillna("") == "A"].copy().reset_index(drop=True)
B_ev = ev[ev["B_transactionType"].fillna("") == "B"].copy().reset_index(drop=True)

for df in (A_tr, B_tr, A_ev, B_ev):
    if "A_id" in df.columns:
        df["A_id"] = df["A_id"].map(norm_id)
    if "B_id" in df.columns:
        df["B_id"] = df["B_id"].map(norm_id

        )

# numeric/date
A_tr["A_amount_f"] = A_tr["A_amount"].map(to_float)
B_tr["B_amount_f"] = B_tr["B_amount"].map(to_float)
A_ev["A_amount_f"] = A_ev["A_amount"].map(to_float)
B_ev["B_amount_f"] = B_ev["B_amount"].map(to_float)

A_tr["A_date"] = A_tr["A_valueDate"].map(to_date)
B_tr["B_date"] = B_tr["B_valueDate"].map(to_date)
A_ev["A_date"] = A_ev["A_valueDate"].map(to_date)
B_ev["B_date"] = B_ev["B_valueDate"].map(to_date)

# text/tokens
A_tr["A_text"] = (A_tr["A_transactionReferences"].fillna("") + " " + A_tr["A_transactionAttributes"].fillna("")).map(clean_text)
B_tr["B_text"] = (B_tr["B_transactionReferences"].fillna("") + " " + B_tr["B_transactionAttributes"].fillna("")).map(clean_text)
A_ev["A_text"] = (A_ev["A_transactionReferences"].fillna("") + " " + A_ev["A_transactionAttributes"].fillna("")).map(clean_text)
B_ev["B_text"] = (B_ev["B_transactionReferences"].fillna("") + " " + B_ev["B_transactionAttributes"].fillna("")).map(clean_text)

A_tr["A_tokens"] = A_tr["A_text"].apply(extract_tokens)
B_tr["B_tokens"] = B_tr["B_text"].apply(extract_tokens)
A_ev["A_tokens"] = A_ev["A_text"].apply(extract_tokens)
B_ev["B_tokens"] = B_ev["B_text"].apply(extract_tokens)

# blocking keys
def add_keys(A_df, B_df):
    A_df["ccy"] = A_df["A_currencyCode"].fillna("").str.strip()
    A_df["acct"] = A_df["A_account"].fillna("").str.strip()
    A_df["dc"] = A_df["A_debitOrCredit"].fillna("").str.strip()
    A_df["amt_r2"] = A_df["A_amount_f"].round(2)

    B_df["ccy"] = B_df["B_currencyCode"].fillna("").str.strip()
    B_df["acct"] = B_df["B_account"].fillna("").str.strip()
    # opposite side expected (validated on train)
    B_df["dc"] = B_df["B_debitOrCredit"].fillna("").apply(opposite)
    B_df["amt_r2"] = B_df["B_amount_f"].round(2)

add_keys(A_tr, B_tr)
add_keys(A_ev, B_ev)

# truth for training labels
Aids_by_mid = A_tr.groupby("matchId")["A_id"].apply(lambda s: set([x for x in s if x])).to_dict()

# token rarity from TRAIN As only (ok)
dfA = Counter()
for toks in A_tr["A_tokens"]:
    dfA.update(set(toks))

# ----------------------------
# CANDIDATE GENERATION (CRITICAL FIX: pass A_df)
# ----------------------------
def build_candidates(B_df, A_df, label=""):
    A_index = A_df[["A_id","ccy","acct","dc","amt_r2","A_amount_f","A_date","A_text","A_tokens","A_allocation","matchId"]].copy()
    A_index["a_row"] = np.arange(len(A_index))

    B_use = B_df[["B_id","ccy","acct","dc","amt_r2","B_amount_f","B_date","B_text","B_tokens","matchId"]].copy()
    B_use["b_row"] = np.arange(len(B_use))

    frames = []
    for off in AMT_CENT_OFFSETS:
        tmp = B_use.copy()
        tmp["amt_key"] = (tmp["amt_r2"] + off).round(2)
        A_tmp = A_index.copy()
        A_tmp["amt_key"] = A_tmp["amt_r2"]

        frames.append(tmp.merge(
            A_tmp,
            on=["ccy","acct","dc","amt_key"],
            how="inner",
            suffixes=("_b","_a")
        ))

    cand = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["B_id","A_id"])
    print(f"[{label}] Initial merged candidates:", len(cand))
    if len(cand) == 0:
        return cand

    # amount filter
    before = len(cand)
    cand["B_amount_f"] = cand["B_amount_f"].astype(float)
    cand["A_amount_f"] = cand["A_amount_f"].astype(float)
    tol = cand["B_amount_f"].apply(amount_tol)
    cand["amt_diff"] = (cand["A_amount_f"] - cand["B_amount_f"]).abs()
    cand = cand[cand["amt_diff"] <= tol]
    print(f"[{label}] After amount filter:", len(cand), "dropped:", before - len(cand))
    if len(cand) == 0:
        return cand

    # date filter
    before = len(cand)
    cand["date_diff"] = (pd.to_datetime(cand["A_date"]) - pd.to_datetime(cand["B_date"])).dt.days.abs()
    cand = cand[(cand["date_diff"].isna()) | (cand["date_diff"] <= DATE_TOL_DAYS)]
    print(f"[{label}] After date filter:", len(cand), "dropped:", before - len(cand))
    if len(cand) == 0:
        return cand

    # quick score + cap
    def quick_score(row):
        btoks = row["B_tokens"] if isinstance(row["B_tokens"], list) else []
        atoks = row["A_tokens"] if isinstance(row["A_tokens"], list) else []
        aset = set(atoks)
        hits = sum(1 for t in btoks if t in aset)
        ss = superstring_hits(btoks, row["A_text"], cap=15)
        return hits * 2 + ss

    before = len(cand)
    cand["quick"] = cand.apply(quick_score, axis=1)
    cand = cand.sort_values(["b_row","quick","amt_diff"], ascending=[True, False, True])
    cand = cand.groupby("b_row").head(MAX_CANDIDATES_PER_B).reset_index(drop=True)
    print(f"[{label}] After cap:", len(cand), "dropped:", before - len(cand))
    return cand

# correct pairing:
cand_tr = build_candidates(B_tr, A_tr, label="TRAIN")
cand_ev = build_candidates(B_ev, A_ev, label="EVAL")

print("Train candidates:", len(cand_tr), "Eval candidates:", len(cand_ev))

# ----------------------------
# TF-IDF (fit on train text only)
# ----------------------------
vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_features=MAX_FEATURES,
    lowercase=False,
    norm="l2"
)

fit_corpus = pd.concat([A_tr["A_text"], B_tr["B_text"]], ignore_index=True).fillna("")
vectorizer.fit(fit_corpus)

A_tr_tfidf = vectorizer.transform(A_tr["A_text"].fillna(""))
B_tr_tfidf = vectorizer.transform(B_tr["B_text"].fillna(""))
A_ev_tfidf = vectorizer.transform(A_ev["A_text"].fillna(""))
B_ev_tfidf = vectorizer.transform(B_ev["B_text"].fillna(""))

# id -> row maps (0..N-1)
A_tr_row = pd.Series(np.arange(len(A_tr)), index=A_tr["A_id"]).to_dict()
B_tr_row = pd.Series(np.arange(len(B_tr)), index=B_tr["B_id"]).to_dict()
A_ev_row = pd.Series(np.arange(len(A_ev)), index=A_ev["A_id"]).to_dict()
B_ev_row = pd.Series(np.arange(len(B_ev)), index=B_ev["B_id"]).to_dict()

def add_tfidf_cos(cand, A_rowmap, B_rowmap, A_mat, B_mat, label=""):
    if len(cand) == 0:
        return cand
    cand = cand.copy()
    cand["a_tfidf_row"] = cand["A_id"].map(A_rowmap)
    cand["b_tfidf_row"] = cand["B_id"].map(B_rowmap)

    before = len(cand)
    cand = cand.dropna(subset=["a_tfidf_row","b_tfidf_row"]).copy()
    dropped = before - len(cand)
    if dropped:
        print(f"[{label}] Dropped unmapped candidates:", dropped)

    if len(cand) == 0:
        return cand

    cand["a_tfidf_row"] = cand["a_tfidf_row"].astype(int)
    cand["b_tfidf_row"] = cand["b_tfidf_row"].astype(int)

    cand["tfidf_cos"] = rowwise_cosine(
        A_mat, B_mat,
        cand["a_tfidf_row"].values,
        cand["b_tfidf_row"].values
    )
    return cand

cand_tr = add_tfidf_cos(cand_tr, A_tr_row, B_tr_row, A_tr_tfidf, B_tr_tfidf, label="TRAIN")
cand_ev = add_tfidf_cos(cand_ev, A_ev_row, B_ev_row, A_ev_tfidf, B_ev_tfidf, label="EVAL")

# guard: if no eval candidates, output all unmatched
if len(cand_ev) == 0:
    print("WARNING: cand_ev empty. Writing all-unmatched submission and exiting.")
    sub = pd.DataFrame({"B_id": B_ev["B_id"].values})
    sub["targetAllocation"] = "[]"
    sub["confidence"] = 0.0
    sub["explanation"] = "UNMATCHED (no candidates after blocking/filters)"
    sub.to_csv(" ", index=False)
    sub.to_csv("submission_with_details.csv", index=False)
    raise SystemExit

# ----------------------------
# FEATURES
# ----------------------------
def compute_features(cand):
    feats = []
    for btoks, atoks, atext, bd, ad, bamt, aamt, cos in zip(
        cand["B_tokens"], cand["A_tokens"], cand["A_text"],
        cand["B_date"], cand["A_date"], cand["B_amount_f"], cand["A_amount_f"], cand["tfidf_cos"]
    ):
        btoks = btoks if isinstance(btoks, list) else []
        atoks = atoks if isinstance(atoks, list) else []
        aset = set(atoks)

        hits = [t for t in btoks if t in aset]
        hit_count = len(hits)
        rare2 = sum(1 for t in hits if dfA.get(t, 999999) <= 2)
        ss = superstring_hits(btoks, atext, cap=20)

        bamt = float(bamt)
        aamt = float(aamt)
        tol = amount_tol(bamt)
        amt_ratio = abs(aamt - bamt) / (tol + 1e-9)

        bd = pd.to_datetime(bd, errors="coerce")
        ad = pd.to_datetime(ad, errors="coerce")
        dd = abs((ad - bd).days) if (pd.notna(bd) and pd.notna(ad)) else DATE_TOL_DAYS
        dd_ratio = dd / max(DATE_TOL_DAYS, 1)

        feats.append([amt_ratio, dd_ratio, hit_count, rare2, ss, float(cos)])
    return np.array(feats, dtype=float)

# labels on training candidates
y = np.array([
    1 if aid in Aids_by_mid.get(mid, set()) else 0
    for mid, aid in zip(cand_tr["matchId_b"], cand_tr["A_id"])
], dtype=int)

X = compute_features(cand_tr)
groups = cand_tr["matchId_b"].astype(str).values

# ----------------------------
# TRAIN + CALIBRATE
# ----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(X, y, groups=groups))

clf = LogisticRegression(max_iter=500)
clf.fit(X[tr_idx], y[tr_idx])

p_va = clf.predict_proba(X[va_idx])[:, 1]
iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(p_va, y[va_idx])

p_va_cal = iso.transform(p_va)
prec, rec, thr = precision_recall_curve(y[va_idx], p_va_cal)

thr_target = None
for i, t in enumerate(thr):
    if prec[i + 1] >= TARGET_PRECISION:
        thr_target = float(t)
        break

print("Target precision:", TARGET_PRECISION, "threshold:", thr_target)
if thr_target is None:
    thr_target = 0.99
    print("WARNING: no threshold hit target precision; using fallback:", thr_target)

# ----------------------------
# SCORE EVAL + 1-to-1 ASSIGNMENT
# ----------------------------
X_ev = compute_features(cand_ev)
p_ev = clf.predict_proba(X_ev)[:, 1]
p_ev_cal = iso.transform(p_ev)

cand_ev = cand_ev.copy()
cand_ev["confidence_raw"] = p_ev
cand_ev["confidence"] = p_ev_cal

cand_keep = cand_ev[cand_ev["confidence"] >= thr_target].copy()
cand_keep = cand_keep.sort_values("confidence", ascending=False)

assigned_A, assigned_B, chosen = set(), set(), []
for r in cand_keep.itertuples(index=False):
    if r.B_id in assigned_B:
        continue
    if r.A_id in assigned_A:
        continue
    assigned_B.add(r.B_id)
    assigned_A.add(r.A_id)
    chosen.append(r)

chosen_df = pd.DataFrame(chosen, columns=cand_keep.columns)
print("Assigned matches:", len(chosen_df), "out of B:", len(B_ev))

# ----------------------------
# OUTPUT FILES
# ----------------------------
sub = pd.DataFrame({"B_id": B_ev["B_id"].values})
sub = sub.merge(
    chosen_df[["B_id","A_id","A_allocation","confidence","confidence_raw","tfidf_cos","amt_diff","date_diff","quick"]],
    on="B_id", how="left"
)

sub["targetAllocation"] = sub["A_allocation"].apply(fmt_alloc_set)
sub["confidence"] = sub["confidence"].fillna(0.0)
sub["explanation"] = np.where(
    sub["confidence"] > 0,
    "TFIDF_CHAR_NGRAM + SUPERSTRING_RESCUE + AMT/DATE_TOL",
    "UNMATCHED (below precision threshold)"
)

submission = sub[["B_id","targetAllocation","confidence","explanation"]]
submission.to_csv("submission.csv", index=False)
sub.to_csv("submission_with_details.csv", index=False)

print("Wrote: submission.csv, submission_with_details.csv")

# ----------------------------
# EVALUATE
# ----------------------------
sol_map = sol.set_index("B_id")["targetAllocation"].to_dict()
pred_map = submission.set_index("B_id")["targetAllocation"].to_dict()

total_B = len(B_ev)
matched_B = sum(1 for pred in pred_map.values() if pred != "[]")

correct = 0
checked = 0
for b, pred_alloc in pred_map.items():
    true_alloc = sol_map.get(b, None)
    if true_alloc is None:
        continue
    if pred_alloc == "[]":
        continue
    checked += 1
    if parse_alloc_set(pred_alloc).issubset(parse_alloc_set(true_alloc)):
        correct += 1

match_rate = matched_B / total_B
match_precision = (correct / checked) if checked else 0.0

print(f"Match rate: {match_rate:.4f} ({matched_B}/{total_B})")
print(f"Match precision (on matched): {match_precision:.4f} ({correct}/{checked})")

/kaggle/input []
/kaggle/input/benchrec-real-world-cash-reconciliation-dataset ['BenchRec_cash_v1.0_train.csv', 'BenchRec_cash_v1.0_solution.csv', 'BenchRec_cash_v1.0_eval.csv']
[TRAIN] Initial merged candidates: 2402651
[TRAIN] After amount filter: 2402651 dropped: 0
[TRAIN] After date filter: 720335 dropped: 1682316
[TRAIN] After cap: 695175 dropped: 25160
[EVAL] Initial merged candidates: 1177347
[EVAL] After amount filter: 1177347 dropped: 0
[EVAL] After date filter: 505067 dropped: 672280
[EVAL] After cap: 497258 dropped: 7809
Train candidates: 695175 Eval candidates: 497258
Target precision: 0.998 threshold: 0.9930748310763451
Assigned matches: 20800 out of B: 32048
Wrote: submission.csv, submission_with_details.csv
Match rate: 0.6490 (20800/32048)
Match precision (on matched): 0.9945 (20686/20800)
